# Supervised fine-tuning (SFT): from scratch and on a real model

This notebook has two parts.

| Part | What you build | Why |
|---|---|---|
| **Part 1: From scratch** | A tiny GPT in pure PyTorch. Pretrain it, then SFT it with a hand-built loss mask. | See every moving part: tokenizer, chat template, label masking, padding, training loop. |
| **Part 2: Real model** | Full SFT of a pretrained 360M-parameter model (`SmolLM2-360M`) on the Alpaca instruction dataset with Hugging Face. | Apply the same ideas to a real LLM and turn a base model into an instruction follower. |

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the free tier is enough). Then run the cells top to bottom.

### The one idea to remember

SFT uses the **same next-token loss as pretraining**. The only difference is that prompt tokens get label **`-100`**, so the model is graded only on the **response**:

```
tokens : <user>  a  d  d  _  2  _  2  <asst>   4    <end>
labels : -100  -100 ...          -100  -100    4    <end>
                └──────── prompt: ignored ───────┘ └ response: learned ┘
```

$$\mathcal{L}_{SFT} = -\frac{1}{|R|}\sum_{t \in R}\log P(x_t \mid x_{<t})$$

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

---
# Part 1: SFT from scratch (pure PyTorch)

Takes about 2 minutes on a T4. Nothing to install: Colab already has PyTorch.

## 1.1 Imports and settings

In [ ]:
import math, random, string, copy, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1.2 Tokenizer with chat special tokens

A character-level tokenizer keeps everything readable. The **special tokens are the chat template**:

- `<user>` marks the start of the prompt
- `<asst>` marks where the answer begins
- `<end>` teaches the model **when to stop**. Without it in the labels, the model never learns to stop.
- `<pad>` fills short sequences in a batch

`IGNORE = -100` is the label value PyTorch's `cross_entropy` skips.

In [ ]:
SPECIALS = ["<pad>", "<user>", "<asst>", "<end>"]
CHARS = list(string.ascii_lowercase + string.ascii_uppercase + string.digits + " :.,")
VOCAB = SPECIALS + CHARS
stoi = {t: i for i, t in enumerate(VOCAB)}
itos = {i: t for t, i in stoi.items()}
PAD, USER, ASST, END = (stoi[s] for s in SPECIALS)
IGNORE = -100

def encode(text): return [stoi[c] for c in text]
def decode(ids):  return "".join(itos[i] for i in ids)

print("vocab size:", len(VOCAB))
print(encode("add 2 2"), "->", decode(encode("add 2 2")))

## 1.3 A tiny GPT, written by hand

- **Causal self-attention:** each token looks only at itself and earlier tokens (triangular mask). That is what makes next-token prediction a fair task.
- **Block:** LayerNorm, then attention with a residual connection, then LayerNorm, then MLP with a residual connection.
- **TinyGPT:** token and position embeddings, 4 blocks, then a linear head that scores every vocabulary entry.

The key SFT line is in `forward`: `ignore_index=IGNORE`. Positions labelled `-100` contribute **nothing** to the loss or the gradients.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)      # Q, K, V in one projection
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("causal", torch.tril(torch.ones(max_len, max_len)).bool())

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, heads, T, head_dim)
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(~self.causal[:T, :T], float("-inf"))   # no peeking ahead
        att = att.softmax(dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm + residual
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=64):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, max_len) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            # >>> THE SFT LINE: targets equal to -100 are skipped <<<
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                                   ignore_index=IGNORE)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20, stop_id=END):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_len:])
            nxt = logits[:, -1].argmax(dim=-1, keepdim=True)    # greedy decoding
            idx = torch.cat([idx, nxt], dim=1)
            if nxt.item() == stop_id:
                break
        return idx


model = TinyGPT(len(VOCAB)).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 1.4 Data

- **Pretraining data:** random words, numbers and punctuation. Plain "web text" with **no instructions** in it.
- **SFT data:** `(prompt, response)` pairs for three skills the base model has never seen: adding numbers, reversing a word and upper-casing a word.

In [ ]:
def rand_word(lo=3, hi=6):
    return "".join(random.choices(string.ascii_lowercase, k=random.randint(lo, hi)))

def pretrain_text(n_chars=200_000):
    parts = []
    while sum(map(len, parts)) < n_chars:
        r = random.random()
        if r < 0.7:   parts.append(rand_word())
        elif r < 0.9: parts.append(str(random.randint(0, 99)))
        else:         parts.append(random.choice([".", ","]))
    return " ".join(parts)

def make_instruction():
    task = random.choice(["add", "reverse", "upper"])
    if task == "add":
        a, b = random.randint(0, 49), random.randint(0, 49)
        return f"add {a} {b}", str(a + b)
    w = rand_word()
    if task == "reverse":
        return f"reverse: {w}", w[::-1]
    return f"upper: {w}", w.upper()

print("pretraining text sample:", pretrain_text(80)[:80])
for _ in range(3):
    print("SFT pair:", make_instruction())

## 1.5 The heart of SFT: building a masked example

1. Wrap the pair in the chat template: `<user> prompt <asst> response <end>`
2. Labels = **`-100` for every prompt token**, real token IDs for the response.
3. Shift by one: position *t* must predict token *t+1*.

`collate` right-pads a batch. Padding gets `<pad>` in the inputs and `-100` in the labels, so padding is ignored too.

In [ ]:
def build_sft_example(prompt, response):
    prompt_ids   = [USER] + encode(prompt) + [ASST]
    response_ids = encode(response) + [END]
    ids    = prompt_ids + response_ids
    labels = [IGNORE] * len(prompt_ids) + response_ids     # mask the prompt
    return ids[:-1], labels[1:]                              # shift for next-token prediction

def collate(batch):
    T = max(len(x) for x, _ in batch)
    xs = torch.full((len(batch), T), PAD, dtype=torch.long)
    ys = torch.full((len(batch), T), IGNORE, dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        xs[i, :len(x)] = torch.tensor(x)
        ys[i, :len(y)] = torch.tensor(y)
    return xs.to(DEVICE), ys.to(DEVICE)

x, y = build_sft_example("add 2 2", "4")
print(f"{'position':>8} | {'input':>6} | {'label (what must be predicted)'}")
for t, (a, b) in enumerate(zip(x, y)):
    print(f"{t:>8} | {itos[a]:>6} | {'-100  (ignored)' if b == IGNORE else itos[b]}")

## 1.6 Training utilities

The same loop is used for both stages:

- **AdamW** with weight decay
- **Warmup, then cosine decay** of the learning rate (warmup avoids large, destabilizing early updates)
- **Gradient clipping** at 1.0 to prevent loss spikes

In [ ]:
def lr_at(step, total, peak, warmup):
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return peak * 0.5 * (1 + math.cos(math.pi * progress))

def train(model, get_batch, steps, peak_lr, warmup, label, log_every=200):
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01)
    model.train()
    losses, t0 = [], time.time()
    for step in range(steps):
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, peak_lr, warmup)
        x, y = get_batch()
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        if step % log_every == 0 or step == steps - 1:
            print(f"[{label}] step {step:5d}  loss {loss.item():.4f}  ({time.time()-t0:.0f}s)")
    return losses

def ask(model, prompt):
    ids = torch.tensor([[USER] + encode(prompt) + [ASST]], device=DEVICE)
    out = model.generate(ids)[0].tolist()[ids.size(1):]
    if out and out[-1] == END:
        out = out[:-1]
    return decode(out)

def accuracy(model, n=300, seed=123):
    rng_state = random.getstate(); random.seed(seed)
    held = [make_instruction() for _ in range(n)]
    random.setstate(rng_state)
    return sum(ask(model, p) == r for p, r in held) / n

TESTS = ["add 12 7", "add 30 15", "reverse: hello", "upper: robot"]

## 1.7 Stage 1: Pretraining (loss on **every** token)

Inputs are a chunk of text, and targets are the same chunk shifted by one character. This produces a **base model**: it knows what text looks like but has never seen an instruction.

In [ ]:
corpus = torch.tensor(encode(pretrain_text()), dtype=torch.long)
BLOCK = 48

def pretrain_batch(bs=64):
    ix = torch.randint(len(corpus) - BLOCK - 1, (bs,))
    x = torch.stack([corpus[i:i + BLOCK] for i in ix])
    y = torch.stack([corpus[i + 1:i + BLOCK + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

pretrain_losses = train(model, pretrain_batch, steps=800, peak_lr=1e-3, warmup=50, label="pretrain")
base_model = copy.deepcopy(model)     # keep a copy of the base model for comparison

print("\nBEFORE SFT (the base model just continues text):")
for p in TESTS:
    print(f"  {p!r:18} -> {ask(model, p)!r}")

## 1.8 Stage 2: Supervised fine-tuning (loss on **response tokens only**)

Same model, same loss function. The only changes are the data, the template, the mask and a **lower learning rate**: SFT adjusts behaviour and should not overwrite what pretraining taught.

In [ ]:
train_set = [build_sft_example(*make_instruction()) for _ in range(20_000)]

def sft_batch(bs=64):
    return collate(random.sample(train_set, bs))

sft_losses = train(model, sft_batch, steps=3000, peak_lr=5e-4, warmup=100, label="sft", log_every=500)

print("\nAFTER SFT:")
for p in TESTS:
    print(f"  {p!r:18} -> {ask(model, p)!r}")
print(f"\nHeld-out exact-match accuracy: base {accuracy(base_model):.1%}  ->  SFT {accuracy(model):.1%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(pretrain_losses); ax[0].set_title("Stage 1: pretraining loss (all tokens)")
ax[1].plot(sft_losses);      ax[1].set_title("Stage 2: SFT loss (response tokens only)")
for a in ax:
    a.set_xlabel("step"); a.set_ylabel("cross-entropy"); a.set_yscale("log"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 1.9 (Optional) Experiment: what if we don't mask the prompt?

Start again from the same base model, but put the prompt tokens in the labels as well. The model now also spends effort learning to predict the **questions**, which are random and unpredictable, so less of the gradient goes into the answers. Compare the accuracy.

In [ ]:
def build_unmasked_example(prompt, response):
    ids = [USER] + encode(prompt) + [ASST] + encode(response) + [END]
    return ids[:-1], ids[1:]                 # every token is a target

random.seed(0)
unmasked_set = [build_unmasked_example(*make_instruction()) for _ in range(20_000)]
no_mask_model = copy.deepcopy(base_model)
train(no_mask_model, lambda bs=64: collate(random.sample(unmasked_set, bs)),
      steps=3000, peak_lr=5e-4, warmup=100, label="sft-no-mask", log_every=1000)

print(f"\nAccuracy WITH prompt masking   : {accuracy(model):.1%}")
print(f"Accuracy WITHOUT prompt masking: {accuracy(no_mask_model):.1%}")

---
# Part 2: SFT on a real pretrained model (Hugging Face)

Now the same recipe on a real LLM:

| Part 1 (toy) | Part 2 (real) |
|---|---|
| Character tokenizer | The model's BPE tokenizer |
| `<user>` / `<asst>` / `<end>` | An Alpaca-style text template plus the model's EOS token |
| Generated toy tasks | `yahma/alpaca-cleaned` (about 52k human-style instructions) |
| TinyGPT, 0.8M parameters | `HuggingFaceTB/SmolLM2-360M`, a **base** model with 360M parameters |
| `-100` mask | Same `-100` mask |

This is **full fine-tuning**: every weight is updated. That fits on a free T4 for a 360M model. For 7B+ models you would switch to **LoRA / QLoRA**, which freezes the base model and trains small adapters.

The training loop is written by hand, without `Trainer` or `SFTTrainer`, so you can see it is the same as Part 1.

Takes about 10 to 15 minutes on a T4.

## 2.1 Install and import

In [ ]:
!pip install -q -U transformers datasets accelerate

In [ ]:
import math, random, time, gc
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup

# Free the GPU memory used by Part 1, if it was run
for _name in ["model", "base_model", "no_mask_model"]:
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

torch.manual_seed(42); random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2.2 Configuration

Change these to experiment. If you hit out-of-memory errors, reduce `BATCH_SIZE` or `MAX_LEN` (and raise `GRAD_ACCUM` to keep the effective batch size).

In [ ]:
MODEL_NAME   = "HuggingFaceTB/SmolLM2-360M"   # a BASE model, not an instruct model
DATASET_NAME = "yahma/alpaca-cleaned"
N_TRAIN      = 4000      # examples used for training (the full set is ~52k)
N_VAL        = 200
MAX_LEN      = 384       # tokens per example (longer examples are truncated)
BATCH_SIZE   = 8
GRAD_ACCUM   = 2         # effective batch = 8 x 2 = 16
EPOCHS       = 1
LR           = 5e-5      # much lower than pretraining: we are nudging, not rebuilding
WARMUP_RATIO = 0.05

# T4 has no bfloat16, so use float16 autocast with a gradient scaler there
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("mixed precision:", AMP_DTYPE)

## 2.3 Load the base model and tokenizer

The weights stay in float32 (needed for stable full fine-tuning). Autocast runs the heavy maths in 16-bit.

Memory estimate: 360M params × (4 B weights + 4 B grads + 8 B AdamW states) ≈ **5.8 GB**, plus activations. That fits on a 15 GB T4.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token     # padding is masked anyway

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(DEVICE)
model.gradient_checkpointing_enable()             # trade compute for activation memory
model.config.use_cache = False                    # the cache is useless during training
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
print("EOS token:", repr(tokenizer.eos_token))

## 2.4 The chat template

A base model has no built-in chat format, so we define one. **Training and inference must use exactly the same template.** A mismatch is one of the most common real-world SFT bugs.

In [ ]:
def format_prompt(instruction, inp=""):
    if inp and inp.strip():
        return ("Below is an instruction that describes a task, paired with an input. "
                "Write a response that completes the request.\n\n"
                f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n")
    return ("Below is an instruction that describes a task. "
            "Write a response that completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n### Response:\n")

print(format_prompt("Name three primary colors."))

## 2.5 Ask the base model *before* SFT

Expect it to ramble, repeat itself or invent new "### Instruction" blocks. It has learned to continue text, not to answer and stop.

In [ ]:
@torch.no_grad()
def ask(instruction, inp="", max_new_tokens=150):
    model.eval()
    model.config.use_cache = True
    enc = tokenizer(format_prompt(instruction, inp), return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.1,
                         eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.pad_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

TEST_PROMPTS = [
    ("Give three tips for staying healthy.", ""),
    ("Translate the sentence into French.", "Where is the train station?"),
    ("Write a Python function that checks whether a number is prime.", ""),
    ("Explain what supervised fine-tuning is in two sentences.", ""),
]

before = {}
for ins, inp in TEST_PROMPTS:
    before[ins] = ask(ins, inp)
    print(f"### {ins} {inp}\n{before[ins]}\n" + "-" * 80)

## 2.6 Load the dataset and build masked examples

This is the same idea as `build_sft_example` in Part 1:

- The prompt and response are tokenized **separately** and then concatenated, so we know exactly where the prompt ends.
- Labels are `-100` for the prompt and real IDs for the response **plus EOS**, so the model learns to stop.
- **No manual shift is needed here.** Hugging Face causal LM models shift the labels internally. In Part 1 we did the shift ourselves.

In [ ]:
raw = load_dataset(DATASET_NAME, split="train").shuffle(seed=42)
raw = raw.select(range(N_TRAIN + N_VAL))
print(raw[0])

def build_example(row):
    prompt_ids   = tokenizer(format_prompt(row["instruction"], row["input"]),
                             add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(row["output"], add_special_tokens=False)["input_ids"]
    response_ids = response_ids + [tokenizer.eos_token_id]       # learn to stop

    input_ids = (prompt_ids + response_ids)[:MAX_LEN]
    labels    = ([-100] * len(prompt_ids) + response_ids)[:MAX_LEN]
    return {"input_ids": input_ids, "labels": labels}

data = raw.map(build_example, remove_columns=raw.column_names)
# drop examples where truncation removed the whole response
data = data.filter(lambda ex: any(l != -100 for l in ex["labels"]))
split = data.train_test_split(test_size=N_VAL, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"train: {len(train_ds)}  val: {len(val_ds)}")

In [ ]:
# Look at one example: which tokens are learned and which are ignored?
ex = train_ds[0]
n_masked = sum(l == -100 for l in ex["labels"])
print(f"total tokens: {len(ex['input_ids'])}   masked (prompt): {n_masked}   "
      f"learned (response): {len(ex['labels']) - n_masked}\n")
print("PROMPT (ignored by the loss):")
print(tokenizer.decode(ex["input_ids"][:n_masked]))
print("\nRESPONSE (learned):")
print(tokenizer.decode(ex["input_ids"][n_masked:]))

## 2.7 Collate: pad each batch

Inputs are padded with the pad token, the attention mask is 0 on padding, and labels are padded with `-100`.

In [ ]:
def collate(batch):
    T = max(len(ex["input_ids"]) for ex in batch)
    input_ids = torch.full((len(batch), T), tokenizer.pad_token_id, dtype=torch.long)
    labels    = torch.full((len(batch), T), -100, dtype=torch.long)
    attn      = torch.zeros((len(batch), T), dtype=torch.long)
    for i, ex in enumerate(batch):
        n = len(ex["input_ids"])
        input_ids[i, :n] = torch.tensor(ex["input_ids"])
        labels[i, :n]    = torch.tensor(ex["labels"])
        attn[i, :n]      = 1
    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

## 2.8 The training loop

The same loop as Part 1, with three additions for a real model:

- **Mixed precision** (`autocast`, plus a `GradScaler` for float16)
- **Gradient accumulation:** sum gradients over `GRAD_ACCUM` mini-batches before each optimizer step
- **Validation loss** after training, to check for overfitting

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    total, n = 0.0, 0
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            loss = model(**batch).loss
        total += loss.item(); n += 1
    model.train()
    return total / max(n, 1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and not USE_BF16))

print(f"val loss before SFT: {evaluate():.4f}")
print(f"optimizer steps: {total_steps}")

model.train()
losses, step, t0 = [], 0, time.time()
for epoch in range(EPOCHS):
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            loss = model(**batch).loss / GRAD_ACCUM      # HF computes the masked loss for us
        scaler.scale(loss).backward()

        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step += 1
            losses.append(loss.item() * GRAD_ACCUM)
            if step % 25 == 0 or step == total_steps:
                print(f"epoch {epoch} step {step:4d}/{total_steps}  loss {losses[-1]:.4f}  "
                      f"lr {scheduler.get_last_lr()[0]:.2e}  ({time.time()-t0:.0f}s)")

print(f"\nval loss after SFT: {evaluate():.4f}")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(losses, alpha=.4, label="per step")
k = 10
if len(losses) >= k:
    smooth = [sum(losses[max(0, i - k + 1):i + 1]) / len(losses[max(0, i - k + 1):i + 1]) for i in range(len(losses))]
    plt.plot(smooth, label=f"moving average ({k})")
plt.xlabel("optimizer step"); plt.ylabel("loss (response tokens only)")
plt.title("SFT training loss"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 2.9 Before vs after

The fine-tuned model should now answer the question directly, follow the format and **stop** at EOS. A 360M model will still make factual mistakes: SFT teaches *behaviour and format*, while knowledge mostly comes from pretraining.

In [ ]:
for ins, inp in TEST_PROMPTS:
    print(f"### {ins} {inp}")
    print(f"--- BEFORE ---\n{before[ins][:400]}")
    print(f"--- AFTER ----\n{ask(ins, inp)}")
    print("=" * 80)

In [ ]:
# Try your own instruction
print(ask("Write a short, friendly email asking my manager for a day off on Friday."))

## 2.10 Save the fine-tuned model

The model is saved to the Colab disk, which is wiped when the runtime ends. Uncomment the Google Drive lines to keep it.

In [ ]:
SAVE_DIR = "smollm2-360m-sft"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("saved to", SAVE_DIR)

# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {SAVE_DIR} /content/drive/MyDrive/

# Reload later with:
# model = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(DEVICE)
# tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

---
## Summary and next steps

**What changed between pretraining and SFT?** Only the data, the template, the `-100` mask and the learning rate. The model, the loss function and the training loop stayed the same.

**Common pitfalls**
- Forgetting EOS in the labels: the model never stops talking.
- Different templates at training and inference time: the outputs degrade badly.
- Learning rate too high or too many epochs: overfitting and catastrophic forgetting. Watch the validation loss.
- Truncation cutting off whole responses: those examples teach nothing, which is why they are filtered out above.

**Things to try**
- Increase `N_TRAIN` to 10k or 20k, or set `EPOCHS = 2`, and compare the outputs.
- Swap `MODEL_NAME` for `HuggingFaceTB/SmolLM2-135M` (faster) or `Qwen/Qwen2.5-0.5B`.
- Re-run Part 2 with `labels = input_ids` (no masking) and compare the validation loss and outputs.
- For bigger models (1B to 8B), keep this data pipeline and switch to LoRA / QLoRA with `peft`, which trains about 1% of the weights.